# Complex file formats with LakeLogic

Nine awkward real-world sources, each run through one contract:

| Source | What makes it hard | Contract features used |
|---|---|---|
| `orders_nested.json` | customer object inside an address object, an array of items | `flatten_nested`, `explode`, `json_extract` |
| `orders_nested.xml` | namespace, attributes, nested customer, repeated `<item>` | the same contract shape as the JSON |
| `customers_fixed_width.txt` | no separators; header and trailer lines; padded numbers; YYYYMMDD dates | `format: fixed_width` with `options.columns` |
| `orders.csv.gz`, `orders_bundle.zip` | compressed; a zip with several files and a README | nothing for `.gz`; optional `archive_member` |
| `orders_eu.csv` | title line, `;`, Latin-1, `10,50`, a line break inside quotes | `options: skip_rows / delimiter / encoding / decimal_comma` |
| `orders_events.avro` | typed Kafka-style records, nested record + array | `format: avro`, `flatten_nested`, `explode` |
| `orders_legacy.xls` | Excel 97-2003, title row, date cells | `format: xls`, `sheet_name`, `header_row` (needs `xlrd`) |
| `orders_multitab.xlsx` | cover tab first, title on row 1, header on row 2, blank row, TOTAL footer | `options: sheet_name / header_row / skip_footer` |

Every source has bad rows on purpose. Each run returns **good** rows and **quarantined** rows with the reason.

**Run it:** from this folder, with LakeLogic installed (`pip install "lakelogic[polars]"`, which includes `openpyxl` and `xlrd`), run all cells.
To use a local LakeLogic checkout instead, set `LAKELOGIC_SRC` below.

In [1]:
import os
import sys
from pathlib import Path

LAKELOGIC_SRC = None  # e.g. r"C:\path\to\lakelogic" to run a local checkout
OLC_SRC = None  # a local open-lakehouse-contract checkout: fixed-width `range` needs it until released
for _src in (OLC_SRC, LAKELOGIC_SRC):
    if _src:
        sys.path.insert(0, _src)

import yaml
import polars as pl
from lakelogic import DataProcessor

pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(45)
pl.Config.set_tbl_width_chars(160)
HERE = Path.cwd()
DATA = HERE / "data"

# (Re)build the sample files
import runpy

runpy.run_path(str(HERE / "make_complex.py"))


def run(contract_yaml: str, show: list[str]):
    contract = yaml.safe_load(contract_yaml)
    good, bad = DataProcessor(engine="polars", contract=contract).run_source()
    print(f"good: {len(good)}   quarantined: {len(bad)}")
    display(good.select([c for c in show if c in good.columns]))
    if len(bad):
        cols = [c for c in show if c in bad.columns] + ["_lakelogic_errors"]
        display(bad.select(cols))
    return good, bad

wrote ['customers_fixed_width.txt', 'orders.csv.gz', 'orders.json', 'orders.jsonl', 'orders.xlsx', 'orders.xml', 'orders_bundle.zip', 'orders_eu.csv', 'orders_events.avro', 'orders_legacy.xls', 'orders_multitab.xlsx', 'orders_nested.json', 'orders_nested.xml', 'scenarios']


## 1. Nested JSON

```json
{"order_id": 5001, "customer": {"name": "Ada", "address": {"city": "London"}},
 "items": [{"sku": "A1", "qty": 2, "price": 9.5}, ...]}
```

- `flatten_nested: true` turns objects into columns: `customer_name`, `customer_address_city`, and so on.
  Arrays stay whole, as JSON text, in their own column (`items`).
- `explode` makes one row per item, and `json_extract` pulls fields out of each item.

Planted problems: order 5003 has `placed_at: "not-a-date"` and an item with `qty: -1`.

In [2]:
JSON_CONTRACT = f"""
version: "1.0"
info: {{title: Nested JSON orders}}
source:
  type: landing
  path: {DATA / "orders_nested.json"}
  format: json
  flatten_nested: true
transformations:
  - explode: {{field: items, output: item}}
  - json_extract: {{field: sku,   source: item, path: "$.sku"}}
  - json_extract: {{field: qty,   source: item, path: "$.qty",   cast: int}}
  - json_extract: {{field: price, source: item, path: "$.price", cast: double}}
model:
  fields:
    - {{name: order_id, type: long, required: true}}
    - {{name: placed_at, type: date}}
    - {{name: customer_name, type: string, required: true}}
    - {{name: customer_id, type: string}}
    - {{name: customer_address_city, type: string}}
    - {{name: customer_address_country, type: string}}
    - {{name: sku, type: string, required: true}}
    - {{name: qty, type: int, required: true}}
    - {{name: price, type: double}}
quality:
  row_rules:
    - {{name: qty_positive, sql: "qty > 0"}}
"""
SHOW = ["order_id", "placed_at", "customer_name", "customer_address_city", "sku", "qty", "price"]
good, bad = run(JSON_CONTRACT, SHOW)

2026-10-07 14:26:18.369 | INFO     | lakelogic.core.processor:run_source:2598 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\file_formats\data\orders_nested.json via polars


2026-10-07 14:26:18.386 | INFO     | lakelogic.core.processor:run:1976 - Run complete [Nested JSON orders] | src=3 rows=4 ok=3 qtn=1 | Expanded: 1 25.00%


good: 3   quarantined: 1


order_id,placed_at,customer_name,customer_address_city,sku,qty,price
i64,date,str,str,str,i32,f64
5001,2026-10-01,"""Ada""","""London""","""A1""",2,9.5
5001,2026-10-01,"""Ada""","""London""","""B2""",1,20.0
5002,2026-10-02,"""Bo""","""Paris""","""C3""",5,1.0


order_id,placed_at,customer_name,customer_address_city,sku,qty,price,_lakelogic_errors
i64,date,str,str,str,i32,f64,list[str]
5003,null,"""Cy""","""Rome""","""A1""",-1,9.5,"[""Type Mismatch: placed_at cannot be cast to date"", ""Rule failed: qty_positive (qty > 0)""]"


## 2. Nested XML

```xml
<export xmlns="urn:shop"><orders>
  <order id="6001" currency="GBP">
    <customer><name>Ada</name><address><city>London</city></address></customer>
    <items><item><sku>A1</sku><qty>2</qty>...</item><item>...</item></items>
```

The reader finds the repeating `<order>` records, drops the namespace, and makes attributes into columns (`id`, `currency`).
Nested elements arrive in the same shape as nested JSON, so the contract is the same shape as the JSON one; the order's attributes become `id` and `currency`.

Planted problem: order 6003 has an item with `<qty>two</qty>`.

In [3]:
XML_CONTRACT = f"""
version: "1.0"
info: {{title: Nested XML orders}}
source:
  type: landing
  path: {DATA / "orders_nested.xml"}
  format: xml
  flatten_nested: true
transformations:
  - explode: {{field: items, output: item}}
  - json_extract: {{field: sku,   source: item, path: "$.sku"}}
  - json_extract: {{field: qty,   source: item, path: "$.qty",   cast: int}}
  - json_extract: {{field: price, source: item, path: "$.price", cast: double}}
model:
  fields:
    - {{name: id, type: long, required: true}}         # the <order id="..."> attribute
    - {{name: currency, type: string}}                 # the <order currency="..."> attribute
    - {{name: placed_at, type: date}}
    - {{name: customer_name, type: string, required: true}}
    - {{name: customer_id, type: string}}
    - {{name: customer_address_city, type: string}}
    - {{name: customer_address_country, type: string}}
    - {{name: sku, type: string, required: true}}
    - {{name: qty, type: int, required: true}}
    - {{name: price, type: double}}
quality:
  row_rules:
    - {{name: qty_positive, sql: "qty > 0"}}
"""
SHOW_XML = ["id", "currency", "placed_at", "customer_name", "customer_address_city", "sku", "qty", "price"]
good, bad = run(XML_CONTRACT, SHOW_XML)

2026-10-07 14:26:18.759 | INFO     | lakelogic.core.processor:run_source:2598 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\file_formats\data\orders_nested.xml via polars


2026-10-07 14:26:18.773 | INFO     | lakelogic.core.processor:run:1976 - Run complete [Nested XML orders] | src=3 rows=6 ok=5 qtn=1 | Expanded: 3 16.67%


good: 5   quarantined: 1


id,currency,placed_at,customer_name,customer_address_city,sku,qty,price
i64,str,date,str,str,str,i32,f64
6001,"""GBP""",2026-10-01,"""Ada""","""London""","""A1""",2,9.5
6001,"""GBP""",2026-10-01,"""Ada""","""London""","""B2""",1,20.0
6002,"""EUR""",2026-10-02,"""Bo""","""Paris""","""C3""",5,1.0
6002,"""EUR""",2026-10-02,"""Bo""","""Paris""","""D4""",1,3.0
6003,"""EUR""",2026-10-03,"""Cy""","""Rome""","""F6""",1,2.0


id,currency,placed_at,customer_name,customer_address_city,sku,qty,price,_lakelogic_errors
i64,str,date,str,str,str,i32,f64,list[str]
6003,"""EUR""",2026-10-03,"""Cy""","""Rome""","""E5""",null,4.0,"[""Rule failed: qty_required (""qty"" IS NOT NULL)"", ""Rule failed: qty_positive (qty > 0)""]"


## 3. Excel: a specific tab, header on row 2

| Row | `Orders` tab |
|---|---|
| 1 | `Q4 order export - generated by finance` (title) |
| 2 | `order_id, customer, country, amount` (header) |
| 3-8 | data, one blank row, then `TOTAL ... 32.5` |

The first tab, `Summary`, is a cover sheet. Planted problems: 7003 has no customer, 7004 has a negative amount.

In [4]:
EXCEL_CONTRACT = f"""
version: "1.0"
info: {{title: Finance order export}}
source:
  type: landing
  path: {DATA / "orders_multitab.xlsx"}
  format: xlsx
  options:
    sheet_name: Orders   # not the first tab
    header_row: 2        # row 1 is a title
    skip_footer: 1       # drop the TOTAL row
model:
  fields:
    - {{name: order_id, type: long, required: true}}
    - {{name: customer, type: string, required: true}}
    - {{name: country, type: string}}
    - {{name: amount, type: double}}
quality:
  row_rules:
    - {{name: amount_not_negative, sql: "amount >= 0"}}
"""
good, bad = run(EXCEL_CONTRACT, ["order_id", "customer", "country", "amount"])

2026-10-07 14:26:18.791 | INFO     | lakelogic.core.processor:run_source:2598 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\file_formats\data\orders_multitab.xlsx via polars


2026-10-07 14:26:18.803 | INFO     | lakelogic.core.processor:run:1976 - Run complete [Finance order export] | src=4 rows=4 ok=2 qtn=2 50.00%


good: 2   quarantined: 2


order_id,customer,country,amount
i64,str,str,f64
7001,"""Ada""","""GB""",10.5
7002,"""Bo""","""FR""",20.0


order_id,customer,country,amount,_lakelogic_errors
i64,str,str,f64,list[str]
7003,null,"""IT""",5.0,"[""Rule failed: customer_required (""customer"" IS NOT NULL)""]"
7004,"""Di""","""ES""",-3.0,"[""Rule failed: amount_not_negative (amount >= 0)""]"


## 4. Fixed-width text

No separators: each field sits at fixed character positions, padded with spaces or zeros, like a mainframe, BACS or payroll file.

```
HDR CUSTOMERS 20261007
000123Ada Lovelace        GB2026100100001050
000124Bob Smith           FR2026100200002000
...
TRL0000004
```

- Each field carries its position: `range: [start, end]`, 0-based and end-exclusive, so `[0, 6]` is characters 1–6.
  (A layout list under `source.options.columns` with 1-based `start` + `width` also works.)
- `record_length: 44`: every record must be 44 characters. A record of any other length is **quarantined**
  with `Line length mismatch: expected 44, got 36`. A file with no line breaks at all is split every 44 characters.
- `skip_rows` / `skip_footer` drop the header and trailer records. Padded numbers like `00001050` cast to 1050.

Planted problems: customer 125 has month 13 and amount `000005X0`; 126 has no name; 127 is cut off.

In [5]:
FIXED_CONTRACT = f"""
version: "1.0"
info: {{title: bronze_mainframe_customers}}
source:
  type: landing
  format: fixed_width
  path: {DATA / "customers_fixed_width.txt"}
  record_length: 44
  encoding: ascii
  skip_rows: 1        # HDR record
  skip_footer: 1      # TRL record
transformations:
  - derive: {{field: signup_date, sql: "try_to_date(CAST(signup_raw AS STRING), 'yyyyMMdd')"}}
model:
  fields:
    - name: customer_id
      range: [0, 6]
      type: long
      required: true
    - name: name
      range: [6, 26]
      type: string
      required: true
    - name: country
      range: [26, 28]
      type: string
    - name: signup_raw
      range: [28, 36]
      type: string
    - name: signup_date
      type: date
      required: true
    - name: amount_pence
      range: [36, 44]
      type: long
      required: true
"""
good, bad = run(FIXED_CONTRACT, ["customer_id", "name", "country", "signup_date", "amount_pence"])

2026-10-07 14:26:18.820 | INFO     | lakelogic.core.processor:run_source:2598 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\file_formats\data\customers_fixed_width.txt via polars


2026-10-07 14:26:18.869 | INFO     | lakelogic.core.processor:run:1976 - Run complete [bronze_mainframe_customers] | src=5 rows=5 ok=2 qtn=3 60.00%


good: 2   quarantined: 3


customer_id,name,country,signup_date,amount_pence
i64,str,str,date,i64
123,"""Ada Lovelace""","""GB""",2026-10-01,1050
124,"""Bob Smith""","""FR""",2026-10-02,2000


customer_id,name,country,signup_date,amount_pence,_lakelogic_errors
i64,str,str,date,i64,list[str]
125,"""Cy Rossi""","""IT""",null,null,"[""Type Mismatch: amount_pence cannot be cast to long"", ""Rule failed: signup_date_required (""signup_date"" IS NOT NULL)"", ""Rule failed: amount_pence_required (""amount_pence"" IS NOT NULL)""]"
126,null,"""ES""",2026-10-04,300,"[""Rule failed: name_required (""name"" IS NOT NULL)""]"
127,"""Di Short""","""ES""",2026-10-04,null,"[""Line length mismatch: expected 44, got 36"", ""Rule failed: amount_pence_required (""amount_pence"" IS NOT NULL)""]"


## 5. Compressed: `.gz`

`orders.csv.gz` is one gzipped CSV. LakeLogic unpacks it to a temp folder, reads it with the normal CSV reader, then deletes the copy.
The format comes from the inner name, so `.json.gz` and `.xml.gz` work the same way.

Planted problem: order 8002 has `amount = n/a`.

In [6]:
GZ_CONTRACT = f"""
version: "1.0"
info: {{title: Gzipped orders}}
source:
  type: landing
  path: {DATA / "orders.csv.gz"}
  format: csv
model:
  fields:
    - {{name: order_id, type: long, required: true}}
    - {{name: customer, type: string, required: true}}
    - {{name: amount, type: double}}
quality:
  row_rules:
    - {{name: amount_not_negative, sql: "amount >= 0"}}
"""
good, bad = run(GZ_CONTRACT, ["order_id", "customer", "amount"])

2026-10-07 14:26:18.881 | INFO     | lakelogic.core.processor:run_source:2598 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\file_formats\data\orders.csv.gz via polars


2026-10-07 14:26:18.926 | INFO     | lakelogic.core.processor:run:1976 - Run complete [Gzipped orders] | src=3 rows=3 ok=2 qtn=1 33.33%


good: 2   quarantined: 1


order_id,customer,amount
i64,str,f64
8001,"""Ada""",10.5
8003,"""Cy""",7.0


order_id,customer,amount,_lakelogic_errors
i64,str,f64,list[str]
8002,"""Bo""",null,"[""Type Mismatch: amount cannot be cast to double"", ""Rule failed: amount_not_negative (amount >= 0)""]"


## 6. Compressed: `.zip` with several files

`orders_bundle.zip` holds `north/orders_north.csv`, `south/orders_south.csv` and a `README.txt`.
Every CSV inside is read and the README is skipped, because only files of the contract's `format` are taken by default.
To pick files yourself, set `archive_member: "north/*.csv"`.

`_source_file` names the archive **and** the file inside it, so a bad row can be traced to its file.
Planted problem: order 9002 has no customer.

In [7]:
ZIP_CONTRACT = f"""
version: "1.0"
info: {{title: Zipped regional orders}}
source:
  type: landing
  path: {DATA / "orders_bundle.zip"}
  format: csv
model:
  fields:
    - {{name: order_id, type: long, required: true}}
    - {{name: customer, type: string, required: true}}
    - {{name: amount, type: double}}
quality:
  row_rules:
    - {{name: amount_not_negative, sql: "amount >= 0"}}
"""
good, bad = run(ZIP_CONTRACT, ["order_id", "customer", "amount", "_source_file"])

2026-10-07 14:26:18.938 | INFO     | lakelogic.core.processor:run_source:2598 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\file_formats\data\orders_bundle.zip via polars


2026-10-07 14:26:18.941 | INFO     | lakelogic.core.processor:run_source:2821 - Archive members come from 2 folders: ['north', 'south']


2026-10-07 14:26:18.949 | INFO     | lakelogic.core.processor:run:1976 - Run complete [Zipped regional orders] | src=3 rows=3 ok=2 qtn=1 33.33%


good: 2   quarantined: 1


order_id,customer,amount,_source_file
i64,str,f64,str
9001,"""Ada""",5.0,"""C:\_Personal\_SaaS\lakelogic\examples\file_fo…"
9101,"""Bo""",8.0,"""C:\_Personal\_SaaS\lakelogic\examples\file_fo…"


order_id,customer,amount,_source_file,_lakelogic_errors
i64,str,f64,str,list[str]
9002,null,6.0,"""C:\_Personal\_SaaS\lakelogic\examples\file_fo…","[""Rule failed: customer_required (""customer"" IS NOT NULL)""]"


## 7. A European CSV

`orders_eu.csv` breaks four defaults at once:

| Quirk | Option |
|---|---|
| a title line above the header | `skip_rows: 1` |
| `;` between fields | `delimiter: ";"` (also `"\\t"` for tab, `"|"` for pipe) |
| Latin-1 text: `François`, `Zoë`, `Jürgen` | `encoding: latin-1` |
| decimal commas: `10,50` | `decimal_comma: true` (only on fields declared as numbers) |

Order 10002's note has a line break inside quotes; it stays one row. Planted problem: order 10003 has a negative amount.

In [8]:
EU_CONTRACT = f"""
version: "1.0"
info: {{title: European order export}}
source:
  type: landing
  path: {DATA / "orders_eu.csv"}
  format: csv
  options:
    skip_rows: 1
    delimiter: ";"
    encoding: latin-1
    decimal_comma: true
model:
  fields:
    - {{name: order_id, type: long, required: true}}
    - {{name: customer, type: string, required: true}}
    - {{name: amount, type: double}}
    - {{name: note, type: string}}
quality:
  row_rules:
    - {{name: amount_not_negative, sql: "amount >= 0"}}
"""
good, bad = run(EU_CONTRACT, ["order_id", "customer", "amount", "note"])

2026-10-07 14:26:18.963 | INFO     | lakelogic.core.processor:run_source:2598 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\file_formats\data\orders_eu.csv via polars


2026-10-07 14:26:18.968 | INFO     | lakelogic.core.processor:run:1976 - Run complete [European order export] | src=3 rows=3 ok=2 qtn=1 33.33%


good: 2   quarantined: 1


order_id,customer,amount,note
i64,str,f64,str
10001,"""François""",10.5,"""livraison rapide"""
10002,"""Zoë""",7.25,"""appeler avant la livraison"""


order_id,customer,amount,note,_lakelogic_errors
i64,str,f64,str,list[str]
10003,"""Jürgen""",-2.0,null,"[""Rule failed: amount_not_negative (amount >= 0)""]"


## 8. Avro (Kafka / Debezium style)

`orders_events.avro` is typed, with a nested `customer` record and an `items` array, like a Kafka export.
Polars reads Avro natively. Nested parts arrive in the same shape as nested JSON, so the same `flatten_nested` + `explode` pattern applies.

Planted problem: order 11003 has no customer name.

In [9]:
AVRO_CONTRACT = f"""
version: "1.0"
info: {{title: Order events}}
source:
  type: landing
  path: {DATA / "orders_events.avro"}
  format: avro
  flatten_nested: true
transformations:
  - explode: {{field: items, output: item}}
  - json_extract: {{field: sku, source: item, path: "$.sku"}}
  - json_extract: {{field: qty, source: item, path: "$.qty", cast: int}}
model:
  fields:
    - {{name: order_id, type: long, required: true}}
    - {{name: customer_name, type: string, required: true}}
    - {{name: customer_tier, type: string}}
    - {{name: sku, type: string, required: true}}
    - {{name: qty, type: int, required: true}}
"""
good, bad = run(AVRO_CONTRACT, ["order_id", "customer_name", "customer_tier", "sku", "qty"])

2026-10-07 14:26:18.980 | INFO     | lakelogic.core.processor:run_source:2598 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\file_formats\data\orders_events.avro via polars


2026-10-07 14:26:18.988 | INFO     | lakelogic.core.processor:run:1976 - Run complete [Order events] | src=3 rows=4 ok=3 qtn=1 | Expanded: 1 25.00%


good: 3   quarantined: 1


order_id,customer_name,customer_tier,sku,qty
i64,str,str,str,i32
11001,"""Ada""","""gold""","""A1""",2
11002,"""Bo""","""silver""","""B2""",1
11002,"""Bo""","""silver""","""C3""",3


order_id,customer_name,customer_tier,sku,qty,_lakelogic_errors
i64,str,str,str,i32,list[str]
11003,null,"""gold""","""D4""",1,"[""Rule failed: customer_name_required (""customer_name"" IS NOT NULL)""]"


## 9. Old Excel `.xls` (97-2003)

`orders_legacy.xls` is a finance-style export: the `Ledger` tab, a title on row 1, the header on row 2, and real Excel date cells.
It needs `xlrd` (`pip install xlrd`, included in `lakelogic[polars]`). The same `sheet_name` and `header_row` options as `.xlsx` apply.

Planted problem: order 12002 has a negative amount.

In [10]:
XLS_CONTRACT = f"""
version: "1.0"
info: {{title: Legacy ledger}}
source:
  type: landing
  path: {DATA / "orders_legacy.xls"}
  format: xls
  options:
    sheet_name: Ledger
    header_row: 2
model:
  fields:
    - {{name: order_id, type: long, required: true}}
    - {{name: booked_on, type: date, required: true}}
    - {{name: amount, type: double}}
quality:
  row_rules:
    - {{name: amount_not_negative, sql: "amount >= 0"}}
"""
good, bad = run(XLS_CONTRACT, ["order_id", "booked_on", "amount"])

2026-10-07 14:26:19.000 | INFO     | lakelogic.core.processor:run_source:2598 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\file_formats\data\orders_legacy.xls via polars


2026-10-07 14:26:19.014 | INFO     | lakelogic.core.processor:run:1976 - Run complete [Legacy ledger] | src=2 rows=2 ok=1 qtn=1 50.00%


good: 1   quarantined: 1


order_id,booked_on,amount
i64,date,f64
12001,2026-10-01,15.0


order_id,booked_on,amount,_lakelogic_errors
i64,date,f64,list[str]
12002,2026-10-02,-4.0,"[""Rule failed: amount_not_negative (amount >= 0)""]"


## Scenario suite

Each scenario writes a small input file with one real-world problem, runs a contract, and **checks** the outcome:
how many rows are good, how many are quarantined, the quarantine reason, or the error raised.
`PASS` means LakeLogic did what it should. The scenarios live in [`scenarios.py`](scenarios.py), so each one is a few readable lines;
their input files are written under `data/scenarios/`. Run this part on its own: it does not depend on the sections above.

In [11]:
import importlib
from loguru import logger

import scenarios as S

importlib.reload(S)  # fresh results each time this cell runs
S.RESULTS.clear()
logger.remove()  # keep the scenario output readable (re-add a sink to see run logs)

### JSON

a missing nested object · ids sent as text · a new field (drift) · an empty items list · a file cut off mid-record

In [12]:
S.json_scenarios()

[PASS] json        a whole customer object is missing                      1 good, 1 quarantined
[PASS] json        ids sent as text; one is not a number                   1 good, 1 quarantined
[PASS] json        a new field appears (schema drift)                      1 good, 0 quarantined
[PASS] json        an order with an empty items list                       1 good, 1 quarantined
[PASS] json        the file is cut off mid-record                          error


### XML

one `<item>` vs many · no `<items>` · a missing id attribute · `&amp;` and CDATA · namespace prefixes · records at different depths (`row_tag`) · a tag never closed

In [13]:
S.xml_scenarios()

[PASS] xml         one <item> where others have several                    3 good, 0 quarantined
[PASS] xml         an order with no <items> at all                         1 good, 1 quarantined
[PASS] xml         the id attribute is missing                             1 good, 1 quarantined
shape: (1, 2)
┌──────────────┬──────┐
│ customer     ┆ sku  │
│ ---          ┆ ---  │
│ str          ┆ str  │
╞══════════════╪══════╡
│ Smith & Sons ┆ A<1> │
└──────────────┴──────┘
[PASS] xml         escaped & and CDATA text survive                        1 good, 0 quarantined


[PASS] xml         namespace prefixes on every tag                         1 good, 0 quarantined
[PASS] xml         records buried at different depths: row_tag             1 good, 1 quarantined
[PASS] xml         a tag is never closed                                   error


### Excel `.xlsx`

header on row 3 · a tab that does not exist · data on the second tab · numbers typed as text · an unexpected TOTAL row · the same with `skip_footer`

In [14]:
S.excel_scenarios()

[PASS] excel       two title rows; header on row 3                         1 good, 1 quarantined
[PASS] excel       the named tab does not exist                            error


[PASS] excel       data is on the second tab                               1 good, 0 quarantined
[PASS] excel       numbers typed as text cells                             1 good, 1 quarantined


[PASS] excel       a TOTAL row nobody told us about                        1 good, 1 quarantined


[PASS] excel       ...and the same file with skip_footer: 1                1 good, 0 quarantined


### CSV

tab · pipe with a pipe inside quotes · a byte-order mark · Latin-1 without and with `encoding` · a short row · `NULL`/`N/A` markers · `1.234,50` · a line break inside quotes

In [15]:
S.csv_scenarios()

[PASS] csv         tab-separated                                           1 good, 0 quarantined
shape: (1, 1)
┌──────────┐
│ customer │
│ ---      │
│ str      │
╞══════════╡
│ A|B Ltd  │
└──────────┘
[PASS] csv         pipe-separated, a pipe inside a quoted name             1 good, 0 quarantined
[PASS] csv         UTF-8 file with a byte-order mark                       1 good, 0 quarantined
[PASS] csv         Latin-1 file, encoding not declared                     error


shape: (1, 1)
┌──────────┐
│ customer │
│ ---      │
│ str      │
╞══════════╡
│ Zoë      │
└──────────┘
[PASS] csv         ...and with encoding: latin-1                           1 good, 0 quarantined
[PASS] csv         a row with a column missing (null amount fails the rule) 1 good, 1 quarantined
[PASS] csv         NULL and N/A written as text                            1 good, 1 quarantined
shape: (1, 1)
┌────────┐
│ amount │
│ ---    │
│ f64    │
╞════════╡
│ 1234.5 │
└────────┘
[PASS] csv         European numbers: 1.234,50                              1 good, 1 quarantined
[PASS] csv         a note with a line break inside quotes                  1 good, 0 quarantined


### Compressed

`.json.gz` · a fake `.gz` · a zip with no CSV · a zip with two folders and different column order · a header-only CSV in a zip

In [16]:
S.compressed_scenarios()

[PASS] compressed  .json.gz                                                1 good, 0 quarantined
[PASS] compressed  a .gz that is not really gzip                           error
[PASS] compressed  a zip with no CSV inside                                error
[PASS] compressed  zip: two folders, columns in a different order          2 good, 0 quarantined
[PASS] compressed  zip: one file lacks a column -> warning, nulls          1 good, 1 quarantined


[PASS] compressed  zip: one CSV is header-only                             1 good, 0 quarantined


### Avro

a later file adds a column · a null nested record · an amount sent as a string

In [17]:
S.avro_scenarios()

shape: (2, 2)
┌──────────┬────────┐
│ order_id ┆ amount │
│ ---      ┆ ---    │
│ i64      ┆ f64    │
╞══════════╪════════╡
│ 1        ┆ null   │
│ 2        ┆ 5.0    │
└──────────┴────────┘


[PASS] avro        a later file adds a column (schema evolution)           2 good, 0 quarantined
[PASS] avro        a null nested record                                    1 good, 1 quarantined
[PASS] avro        amount sent as a string                                 1 good, 1 quarantined


### Fixed-width

a short record · a long record · no line breaks · Windows line endings · EBCDIC · a letter in a number · a sender that trims trailing spaces · implied decimals

In [18]:
S.fixed_width_scenarios()

[PASS] fixed_width a record cut short                                      1 good, 1 quarantined


[PASS] fixed_width a record too long                                       1 good, 1 quarantined
[PASS] fixed_width no line breaks (mainframe fixed-length)                 2 good, 0 quarantined
[PASS] fixed_width Windows line endings (CRLF)                             2 good, 0 quarantined


[PASS] fixed_width EBCDIC from a mainframe (cp037)                         1 good, 0 quarantined
[PASS] fixed_width a letter in the amount                                  0 good, 1 quarantined


[PASS] fixed_width sender trims trailing spaces                            0 good, 2 quarantined
shape: (1, 1)
┌────────┐
│ amount │
│ ---    │
│ f64    │
╞════════╡
│ 10.5   │
└────────┘
[PASS] fixed_width implied decimals: 00001050 -> 10.50                     1 good, 1 quarantined
shape: (1, 1)
┌────────────┐
│ value_date │
│ ---        │
│ date       │
╞════════════╡
│ 2026-10-07 │
└────────────┘
[PASS] fixed_width dates as YYYYMMDD; month 13 is flagged                  1 good, 1 quarantined


### Excel `.xls`

a 97-2003 workbook with a named tab and header on row 2

In [19]:
S.xls_scenarios()

[PASS] xls         Excel 97-2003, named tab, header on row 2               1 good, 1 quarantined


### Summary

Every scenario, what happened, and why a `FAIL` failed.

In [20]:
with pl.Config(tbl_rows=100, fmt_str_lengths=80, tbl_width_chars=220):
    display(S.summary().select(["status", "format", "scenario", "result", "detail", "why"]))


46 of 46 scenarios behaved as expected.


status,format,scenario,result,detail,why
str,str,str,str,str,str
"""PASS""","""json""","""a whole customer object is missing""","""1 good, 1 quarantined""","""Rule failed: customer_name_required (""customer_name"" IS NOT NULL)""",""""""
"""PASS""","""json""","""ids sent as text; one is not a number""","""1 good, 1 quarantined""","""Rule failed: order_id_required (""order_id"" IS NOT NULL); Type Mismatch: order_id…",""""""
"""PASS""","""json""","""a new field appears (schema drift)""","""1 good, 0 quarantined""","""""",""""""
"""PASS""","""json""","""an order with an empty items list""","""1 good, 1 quarantined""","""Rule failed: sku_required (""sku"" IS NOT NULL)""",""""""
"""PASS""","""json""","""the file is cut off mid-record""","""error""","""JSONDecodeError: Expecting value: line 1 column 30 (char 29)""",""""""
"""PASS""","""xml""","""one <item> where others have several""","""3 good, 0 quarantined""","""""",""""""
"""PASS""","""xml""","""an order with no <items> at all""","""1 good, 1 quarantined""","""Rule failed: qty_required (""qty"" IS NOT NULL); Rule failed: sku_required (""sku"" …",""""""
"""PASS""","""xml""","""the id attribute is missing""","""1 good, 1 quarantined""","""Rule failed: id_required (""id"" IS NOT NULL)""",""""""
"""PASS""","""xml""","""escaped & and CDATA text survive""","""1 good, 0 quarantined""","""""",""""""


## What happened to every row

Nothing is lost. Each input row is either **good** or **quarantined** with its reasons in `_lakelogic_errors`.
The JSON and XML sources count *items*, because `explode` makes one row per item.

Try it yourself: remove `skip_footer: 1` and the TOTAL row is quarantined (its `order_id` is not a number).
Or set `sheet_name: Summary` and the run fails, because that tab has no `order_id` column.